In [2]:
#04_modelling_evalutation.ipynb

In [3]:
!git clone https://github.com/Aymanberri/RealEstate_Analytics.git


Cloning into 'RealEstate_Analytics'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (196/196), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 196 (delta 111), reused 93 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (196/196), 342.93 KiB | 2.38 MiB/s, done.
Resolving deltas: 100% (111/111), done.


In [4]:
%cd RealEstate_Analytics
!ls

/content/RealEstate_Analytics
app  architecture.md  data  notebooks  README.md  requirements.txt


---

# Notebook 04 — Modeling & Evaluation

This notebook trains and evaluates machine learning models
to predict annual rent prices for apartments in JVC, Dubai.

## Imports

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Load dataset

The csv we prepared in step 03.

In [6]:
DATA_PATH = "data/jvc_apartments_ml.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (951, 51)


,title,price,frequency,bedrooms,bathrooms,area,location,url,price_clean,price_yearly_aed,...,building_grouped_Westview Garden,district_JVC District 11,district_JVC District 12,district_JVC District 13,district_JVC District 14,district_JVC District 15,district_JVC District 16,district_JVC District 17,district_JVC District 18,district_Unknown
0,1BR Apartment for Rent | Balcony & Pool | JVC,"74,000",yearly,1,2,905 sqft,"AAA Residence, JVC District 13, Jumeirah Villa...",https://www.bayut.com/property/details-9398172...,74000,74000,...,False,False,False,True,False,False,False,False,False,False
1,1 B/R with Balcony | Pool & Gym | JVC,"69,000",yearly,1,1,883 sqft,"Emerald Tower, JVC District 18, Jumeirah Villa...",https://www.bayut.com/property/details-4864285...,69000,69000,...,False,False,False,False,False,False,False,False,True,False
2,"Binghatti Phoenix, Jumeirah Village Circle, Dubai","89,990",yearly,1,2,826 sqft,"Binghatti Phoenix, JVC District 13, Jumeirah V...",https://www.bayut.com/property/details-1347054...,89990,89990,...,False,False,False,True,False,False,False,False,False,False
3,Converted into 2BR | Private Garden | Furnished,"140,000",yearly,1,2,"1,133 sqft","Signature Livings South, Signature Livings, JV...",https://www.bayut.com/property/details-1330702...,140000,140000,...,False,False,False,False,False,False,False,False,False,False
4,Spacious 1Br | Prime Location | JVC,"75,000",yearly,1,2,925 sqft,"Reef Residence, JVC District 13, Jumeirah Vill...",https://www.bayut.com/property/details-1366409...,75000,75000,...,False,False,False,True,False,False,False,False,False,False


In [7]:
print(df.columns)

Index(['title', 'price', 'frequency', 'bedrooms', 'bathrooms', 'area',
       'location', 'url', 'price_clean', 'price_yearly_aed', 'bedrooms_clean',
       'bathrooms_clean', 'area_clean', 'area_per_bedroom',
       'bathrooms_per_bedroom', 'log_area', 'log_price', 'building',
       'community', 'property_type_penthouse', 'property_type_townhouse',
       'property_type_villa', 'building_grouped_Binghatti Corner',
       'building_grouped_Binghatti Crest',
       'building_grouped_Binghatti Heights',
       'building_grouped_Binghatti House',
       'building_grouped_Binghatti Phantom',
       'building_grouped_Binghatti Phoenix',
       'building_grouped_Binghatti Royale',
       'building_grouped_Bloom Heights 1, Bloom Heights',
       'building_grouped_DAMAC Ghalia', 'building_grouped_Fortunato',
       'building_grouped_Imperial Tower', 'building_grouped_Laya Residences',
       'building_grouped_Other', 'building_grouped_Pearl House 2',
       'building_grouped_Reef Residence', 

## Define Train (X) and Target (y) features

In [8]:
# We must drop columns that are useless (memory usage) and derived from the `price_yearly_aed` (data leakage)
# this step is not cleaning or preprocessesing, this is why I put it here. In ML we now choose only what we need.
DROP_COLS = [
    "title",
    "price",      # Derived features cause data leakage
    "price_clean", # Derived
    "frequency",
    "url",
    "location",
    "area",
    "bedrooms",
    "bathrooms",

    "price_yearly_aed",
    "log_price" # Derived
]

X = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
y = df["price_yearly_aed"] # TARGET

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (951, 40)
y shape: (951,)


## Test/Train Split

Here is where we will split the data. We will then train the models on one part (80%) and the rest is for testing the model.

```
TRAIN / TEST SPLIT
   │
   ├──► X_train, y_train  ──► MODEL TRAINING
   │
   └──► X_test, y_test    ──► MODEL EVALUATION
                                │
                                ▼
                          METRICS (MAE, RMSE, R²)
                                │
                                ▼
                        MODEL COMPARISON
                                │
                                ▼
                       BEST MODEL SELECTED
                                │
                                ▼
                    SAVE MODEL / USE FOR PREDICTION
```

_The model learns from (X_train, y_train), predicts using X_test, and we evaluate by comparing predictions to y_test._

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42 # controls randomness, and keeps results reproducible
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)




Train shape: (760, 40) (760,)
Test shape: (191, 40) (191,)


## Baseline model

we create a baseline model to check how a simple baseline performs.    
Metrics used to evaluate this model is the `MAE`, `RMSE`, and `R2`


In [12]:
# Baseline predictions
y_pred_baseline = np.full_like(y_test, y_train.mean(), dtype=np.float64)

mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
mse_baseline = mean_squared_error(y_test, y_pred_baseline)
rmse_baseline = np.sqrt(mse_baseline)
r2_baseline = r2_score(y_test, y_pred_baseline)

print(f"Baseline MAE: {mae_baseline:.0f}, RMSE: {rmse_baseline:.0f}, R²: {r2_baseline:.2f}")

Baseline MAE: 33740, RMSE: 48103, R²: -0.00


Understanding the metrics:

So for our case, we will try to predic yearly rent, which is a number. The model makes guesses, Then we will have to check how well it guesses...     

     
`MAE`: On avg, how far is the guess from the real number?
- Formula: Take the difference between prediction and real value, make it positive, average it.
- ```
    Example:
    Real rents = [1000, 2000, 3000]
    Predicted = [1200, 1800, 3100]
    Errors = [200, 200, 100] → MAE = (200+200+100)/3 = 167
    ```
- LOWER is better

    

`RSME`: Similar to `MAE`, but we square the errors first, then take square root.
- Big mistakes are punished more.
- ```
    Example:
    Errors = [200, 200, 100]
    Squared = [40000, 40000, 10000] → mean = 30000 → sqrt = 173.2
    ```
- LOWER is better

   
`R^2`: How much better is the model than just guessing the mean?
- ```
    R² = 0 → model explains 0% of the variation (just guessing the mean)
    R² = 0.7 → model explains 70% of why rents differ
    R² = 1 → perfect model
    R² < 0 → your model is worse than guessing the mean
    ```

- HIGHER is better (closer to 1)


